# NB24 · Context Managers

`advanced/` · notebook 24 of 32 · 28 questions + a mini-project

You've used `with open(...) as f:` since NB13. This notebook shows what
`with` actually does, and how to write your own context managers for
anything that needs a guaranteed "after" step: closing, unlocking,
restoring, rolling back, timing.

The protocol is two methods. `with thing as x:` calls `thing.__enter__()`
and binds its return value to `x`. When the block ends, **however** it
ends (normally, with `return`, or with an exception), Python calls
`thing.__exit__(exc_type, exc, tb)`. If `__exit__` returns `True`, the
exception is swallowed. That's the whole contract.

`contextlib` then gives you shortcuts: `@contextmanager` turns a generator
(NB22) into a context manager, and helpers like `suppress`,
`redirect_stdout` and `ExitStack` cover the common jobs. Timers here use a
*fake clock* so their output never changes.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB23: decorators
- [ ] NB12: exceptions

## What you'll practise

- The with protocol: __enter__ and __exit__
- Exception info and suppression (returning True)
- contextlib.contextmanager, suppress, redirect_stdout, closing, nullcontext
- ExitStack for a varying number of resources
- Patterns: temporary changes, timers, transactions
- Parenthesised multi-item with (3.10+)

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB24-Q01 · L1 · The with protocol · Write

Write a class `Session` whose `__init__` stores a user name. `__enter__`
prints `login <name>` and returns the session; `__exit__` prints
`logout <name>`. Use it in a `with` block that prints `working`.

```text
login ann
working
logout ann
```

<details><summary><b>Hint</b></summary>

`__exit__` must accept three extra parameters (`exc_type, exc, tb`), even if you don't use them. Python always passes them.

</details>

<details><summary><b>Expected output</b></summary>

```text
login ann
working
logout ann
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Session:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print("login", self.name)
        return self

    def __exit__(self, exc_type, exc, tb):
        print("logout", self.name)

with Session("ann"):
    print("working")
```

`with Session("ann"):` creates the object, calls `__enter__` before the block and `__exit__` after it. The three arguments to `__exit__` describe an exception if one happened; here there was none, so they are all `None`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Session:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print(f"login {self.name}")
        return self

    def __exit__(self, exc_type, exc, tb):
        print(f"logout {self.name}")
        return False

with Session("ann") as session:
    print("working")
```

`return False` says out loud "I don't swallow exceptions". Returning nothing (`None`) means the same, but the explicit version stops readers wondering. Returning `self` from `__enter__` is the usual choice, so `as session` gives you the object.

</details>

In [ ]:
# your code here


### NB24-Q02 · L1 · The with protocol · Predict

`__enter__` has no `return` line. Predict the output.

**What does this print?**

```python
class Connection:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print("open", self.name)

    def __exit__(self, exc_type, exc, tb):
        print("close", self.name)

with Connection("db") as conn:
    print("inside:", conn)
```

<details><summary><b>Hint</b></summary>

The name after `as` is not the object after `with`. It's whatever `__enter__` returns.

</details>

<details><summary><b>Expected output</b></summary>

```text
open db
inside: None
close db
```

</details>

<details><summary><b>Solution</b></summary>

`as conn` binds the **return value of `__enter__`**, not the `Connection` object. With no `return`, that's `None`. The block still runs, and `__exit__` still closes the connection afterwards.

</details>

<details><summary><b>Senior dev solution</b></summary>

Forgetting `return self` is a classic bug that shows up later as `AttributeError: 'NoneType' object has no attribute ...`. Some managers return something else on purpose: `open()` returns the file, and a lock returns `True`.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB24-Q03 · L1 · contextmanager · Write

Use `contextlib.contextmanager` to write `tag(name)`: it prints `<name>`
before the block and `</name>` after it. Use it to wrap a line that
prints `Hello`.

```text
<p>
Hello
</p>
```

<details><summary><b>Hint</b></summary>

Write a generator function (NB22): print the opening tag, `yield`, then print the closing tag. Put `@contextmanager` above it.

</details>

<details><summary><b>Expected output</b></summary>

```text
<p>
Hello
</p>
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def tag(name):
    print(f"<{name}>")
    yield
    print(f"</{name}>")

with tag("p"):
    print("Hello")
```

`@contextmanager` turns the generator into a context manager. Everything before `yield` is the `__enter__` part, the `with` block runs while the generator is paused at `yield`, and everything after is the `__exit__` part.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def tag(name):
    print(f"<{name}>")
    try:
        yield
    finally:
        print(f"</{name}>")

with tag("p"):
    print("Hello")
```

Seniors always put the code after `yield` in `finally`. Without it, an exception in the block skips the closing tag entirely (Q11 shows the bug). Same output here, but only this version is safe.

</details>

In [ ]:
# your code here


---

## L2 · Easy

### NB24-Q04 · L2 · Patterns: timers · Write

Write a class `Timer` that takes a `clock` function. `__enter__` records
the start time and returns the timer; `__exit__` stores the duration in
`self.elapsed`. Time the given work with `fake_clock`, then print the
result and the time with 2 decimal places.

```text
total: 499500
took 1.75s
```

**Starting code (already in the cell below):**

```python
readings = iter([5.0, 6.75])

def fake_clock():
    return next(readings)
```

<details><summary><b>Hint</b></summary>

Store `self.start = self.clock()` in `__enter__`. In `__exit__`, subtract it from a fresh `self.clock()`. Use `with Timer(fake_clock) as t:` so you can read `t.elapsed` after the block.

</details>

<details><summary><b>Expected output</b></summary>

```text
total: 499500
took 1.75s
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Timer:
    def __init__(self, clock):
        self.clock = clock
        self.elapsed = None

    def __enter__(self):
        self.start = self.clock()
        return self

    def __exit__(self, exc_type, exc, tb):
        self.elapsed = self.clock() - self.start

with Timer(fake_clock) as t:
    total = sum(range(1000))

print("total:", total)
print(f"took {t.elapsed:.2f}s")
```

The timer reads the clock twice: `5.0` on entry and `6.75` on exit, so `elapsed` is `1.75`. Because `__enter__` returns `self`, `t` is the timer, and its attribute is still there after the block ends. Variables created inside a `with` block (like `total`) also stay available afterwards.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import time

class Timer:
    def __init__(self, clock=time.perf_counter):
        self.clock = clock
        self.elapsed = None

    def __enter__(self):
        self.start = self.clock()
        return self

    def __exit__(self, *exc_info):
        self.elapsed = self.clock() - self.start
        return False

with Timer(clock=fake_clock) as t:
    total = sum(range(1000))

print(f"total: {total}")
print(f"took {t.elapsed:.2f}s")
```

In real code `time.perf_counter` (NB15) is the default, and tests inject a fake clock. `*exc_info` is a tidy way to accept the three exception arguments when you don't need them. `__exit__` runs even if the block crashes, so a failed run is still timed.

</details>

In [ ]:
readings = iter([5.0, 6.75])

def fake_clock():
    return next(readings)

# your code here


### NB24-Q05 · L2 · Exception info · Predict

`__exit__` reports what it was given. Predict the output.

**What does this print?**

```python
class Watch:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        name = exc_type.__name__ if exc_type else None
        print("exit sees:", name)

with Watch():
    print("fine")

try:
    with Watch():
        int("abc")
except ValueError:
    print("caught outside")
```

<details><summary><b>Hint</b></summary>

With no exception, all three arguments are `None`. With one, `exc_type` is the exception's class. Does `__exit__` stop the exception?

</details>

<details><summary><b>Expected output</b></summary>

```text
fine
exit sees: None
exit sees: ValueError
caught outside
```

</details>

<details><summary><b>Solution</b></summary>

The first block ends normally, so `exc_type` is `None`. In the second, `int("abc")` raises `ValueError`; Python calls `__exit__` with the class, the exception object and the traceback. `__exit__` returns `None` (falsy), so the exception carries on and the outer `except` catches it.

</details>

<details><summary><b>Senior dev solution</b></summary>

`__exit__` sees every exception, which makes it the right place for cleanup and logging. Whether the exception continues is decided by its return value alone (Q10).

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB24-Q06 · L2 · contextlib.suppress · Fix

The basket total should skip items with no price, so it should print
`1.25`. It prints `0` because `suppress` is hiding a second, real bug.
Fix both.

**Buggy code (copied into the cell below):**

```python
from contextlib import suppress

prices = {"apple": 0.5, "pear": 0.75}
basket = ["apple", "kiwi", "pear"]

total = 0
for item in basket:
    with suppress(Exception):
        total += pricse[item]
print(total)
```

<details><summary><b>Hint</b></summary>

Run the line inside the `with` on its own. Which error does it raise? Which error did the author actually mean to ignore?

</details>

<details><summary><b>Expected output</b></summary>

```text
1.25
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import suppress

prices = {"apple": 0.5, "pear": 0.75}
basket = ["apple", "kiwi", "pear"]

total = 0
for item in basket:
    with suppress(KeyError):
        total += prices[item]
print(total)
```

`pricse` is a typo, so every line raised `NameError`, and `suppress(Exception)` silently swallowed all three. Suppressing only `KeyError`, the one error you expect (a missing price), would have made the typo crash loudly. Fixing the typo then gives `0.5 + 0.75 = 1.25`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
prices = {"apple": 0.5, "pear": 0.75}
basket = ["apple", "kiwi", "pear"]

total = sum(prices.get(item, 0) for item in basket)
print(total)
```

Suppressing by accident is the classic context-manager trap: always name the narrowest exception. Here `dict.get` with a default avoids the exception altogether, which is clearer still. `suppress` shines when there is no such method, e.g. deleting a file that may not exist.

</details>

In [ ]:
from contextlib import suppress

prices = {"apple": 0.5, "pear": 0.75}
basket = ["apple", "kiwi", "pear"]

total = 0
for item in basket:
    with suppress(Exception):
        total += pricse[item]
print(total)


### NB24-Q07 · L2 · contextlib.redirect_stdout · Write

`noisy_report()` prints its results instead of returning them. Capture
everything it prints into an `io.StringIO` using
`contextlib.redirect_stdout`, then print how many lines you captured
and the lines in upper case, joined by ` | `.

```text
captured 2 lines
ROWS: 3 | ERRORS: 0
```

**Starting code (already in the cell below):**

```python
def noisy_report():
    print("rows: 3")
    print("errors: 0")
```

<details><summary><b>Hint</b></summary>

`with redirect_stdout(buffer):` sends every `print()` inside the block into `buffer`. Afterwards, `buffer.getvalue()` gives you the text.

</details>

<details><summary><b>Expected output</b></summary>

```text
captured 2 lines
ROWS: 3 | ERRORS: 0
```

</details>

<details><summary><b>Solution</b></summary>

```python
import io
from contextlib import redirect_stdout

buffer = io.StringIO()
with redirect_stdout(buffer):
    noisy_report()

lines = buffer.getvalue().splitlines()
print("captured", len(lines), "lines")
print(" | ".join(line.upper() for line in lines))
```

`print()` writes to `sys.stdout`. `redirect_stdout` swaps `sys.stdout` for your buffer on entry and puts the real one back on exit, even if the block crashes. So the two prints land in the buffer, and your own prints after the block reach the screen again.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import io
from contextlib import redirect_stdout

with redirect_stdout(io.StringIO()) as buffer:
    noisy_report()

lines = buffer.getvalue().splitlines()
print(f"captured {len(lines)} lines")
print(" | ".join(map(str.upper, lines)))
```

`redirect_stdout` returns the target from `__enter__`, so `as buffer` saves a line. It's handy for testing code that prints and for silencing chatty libraries. When you own the code, return values instead of printing, and nothing needs capturing.

</details>

In [ ]:
def noisy_report():
    print("rows: 3")
    print("errors: 0")

# your code here


### NB24-Q08 · L2 · The with protocol · Error

`__exit__` runs, but does it stop the error? What is printed, and which error is raised?

**Which error does this raise?**

```python
class Cleanup:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        print("cleanup done")

with Cleanup():
    rate = 10 / 0
    print("never printed")
```

<details><summary><b>Hint</b></summary>

`__exit__` runs on the way out, whatever happened. What does it return?

</details>

<details><summary><b>Expected output</b></summary>

```text
cleanup done
ZeroDivisionError
```

</details>

<details><summary><b>Solution</b></summary>

`10 / 0` raises `ZeroDivisionError`. The rest of the block is skipped, `__exit__` runs and prints `cleanup done`, then returns `None`. `None` is falsy, so the exception keeps travelling and the program stops with `ZeroDivisionError`. Cleanup **and** a visible error: the normal, correct behaviour.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Cleanup:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        print("cleanup done")
        return False

try:
    with Cleanup():
        rate = 10 / 0
except ZeroDivisionError:
    print("handled by the caller")
```

A context manager's job is cleanup, not error handling. The caller decides what an error means, with an ordinary `try`/`except` around the `with`.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB24-Q09 · L2 · contextlib.nullcontext · Write

`run_job(name, audit)` should wrap its work in `audit_block(name)` only
when `audit` is `True`. Write it **without** duplicating the work line in
an `if`/`else`: choose the context manager first, then use one `with`.
Use `contextlib.nullcontext()` as the "do nothing" manager. Call
`run_job("backup", True)` and `run_job("cleanup", False)`.

```text
audit start: backup
running backup
audit end: backup
running cleanup
```

**Starting code (already in the cell below):**

```python
from contextlib import contextmanager

@contextmanager
def audit_block(name):
    print("audit start:", name)
    try:
        yield
    finally:
        print("audit end:", name)
```

<details><summary><b>Hint</b></summary>

`manager = audit_block(name) if audit else nullcontext()`, then `with manager:`.

</details>

<details><summary><b>Expected output</b></summary>

```text
audit start: backup
running backup
audit end: backup
running cleanup
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import nullcontext

def run_job(name, audit):
    manager = audit_block(name) if audit else nullcontext()
    with manager:
        print("running", name)

run_job("backup", True)
run_job("cleanup", False)
```

`nullcontext()` is a context manager that does nothing on entry or exit. It lets one `with` statement handle both cases, so the real work is written once. Context managers are ordinary objects, so you can pick one with an expression and store it in a variable.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from contextlib import nullcontext

def run_job(name, audit=False):
    with audit_block(name) if audit else nullcontext():
        print(f"running {name}")

run_job("backup", audit=True)
run_job("cleanup", audit=False)
```

Passing `audit=True` by keyword makes the call readable; a bare `True` makes readers look up the signature. `nullcontext(value)` can also hand back a value from `as`, e.g. an already-open file the function must not close.

</details>

In [ ]:
from contextlib import contextmanager

@contextmanager
def audit_block(name):
    print("audit start:", name)
    try:
        yield
    finally:
        print("audit end:", name)

# your code here


---

## L3 · Intermediate

### NB24-Q10 · L3 · Exception suppression · Predict

This `__exit__` returns `True`. Predict the output.

**What does this print?**

```python
class Ignore:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        print("exit got", exc_type.__name__ if exc_type else None)
        return True

stock = {"pen": 4}
with Ignore():
    print("step 1")
    count = stock["ink"]
    print("step 2")
print("after the block")
```

<details><summary><b>Hint</b></summary>

Returning `True` from `__exit__` tells Python the exception has been dealt with. Where does the program carry on from?

</details>

<details><summary><b>Expected output</b></summary>

```text
step 1
exit got KeyError
after the block
```

</details>

<details><summary><b>Solution</b></summary>

`stock["ink"]` raises `KeyError`, so `step 2` is skipped. `__exit__` receives the exception and returns `True`, which **swallows** it. The program continues after the `with` block as if nothing happened. The block itself never resumes.

</details>

<details><summary><b>Senior dev solution</b></summary>

A manager that returns `True` unconditionally hides every bug inside its block, typos included. Seniors only return `True` for specific, expected exception types, which is what `contextlib.suppress(KeyError)` does.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB24-Q11 · L3 · contextmanager · Fix

`opened` should always print `close <name>`, even when the block fails.
Here the `ValueError` skips it. Fix the context manager so the output is:

```text
open data.csv
reading data.csv
close data.csv
error: bad row
```

**Buggy code (copied into the cell below):**

```python
from contextlib import contextmanager

@contextmanager
def opened(name):
    print("open", name)
    yield name
    print("close", name)

try:
    with opened("data.csv") as f:
        print("reading", f)
        raise ValueError("bad row")
except ValueError as e:
    print("error:", e)
```

<details><summary><b>Hint</b></summary>

When the block raises, the exception is thrown *into* the generator at the `yield` line. What makes code run no matter how a block ends?

</details>

<details><summary><b>Expected output</b></summary>

```text
open data.csv
reading data.csv
close data.csv
error: bad row
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def opened(name):
    print("open", name)
    try:
        yield name
    finally:
        print("close", name)

try:
    with opened("data.csv") as f:
        print("reading", f)
        raise ValueError("bad row")
except ValueError as e:
    print("error:", e)
```

`@contextmanager` re-raises the block's exception at the `yield`. Without `try`, the exception leaves the generator straight away and the cleanup line never runs. `try`/`finally` around the `yield` guarantees it. The exception still reaches the caller, because `finally` doesn't swallow it.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def opened(name):
    print(f"open {name}")
    try:
        yield name
    finally:
        print(f"close {name}")

try:
    with opened("data.csv") as f:
        print(f"reading {f}")
        raise ValueError("bad row")
except ValueError as e:
    print(f"error: {e}")
```

Same fix, and it's the rule seniors follow without thinking: in a `@contextmanager`, the `yield` always sits inside `try`/`finally`. A missing `finally` is how files, locks and connections leak.

</details>

In [ ]:
from contextlib import contextmanager

@contextmanager
def opened(name):
    print("open", name)
    yield name
    print("close", name)

try:
    with opened("data.csv") as f:
        print("reading", f)
        raise ValueError("bad row")
except ValueError as e:
    print("error:", e)


### NB24-Q12 · L3 · contextlib.closing · Write

`Connection` has a `close()` method but no `__enter__`/`__exit__`. Use
`contextlib.closing` to make sure it's closed after use. Open
`Connection("orders")`, print the result of `conn.query("SELECT 1")`,
and let `closing` close it.

```text
result of SELECT 1
connection orders closed
```

**Starting code (already in the cell below):**

```python
class Connection:
    def __init__(self, name):
        self.name = name

    def query(self, sql):
        return f"result of {sql}"

    def close(self):
        print(f"connection {self.name} closed")
```

<details><summary><b>Hint</b></summary>

`closing(thing)` returns `thing` from `as`, and calls `thing.close()` when the block ends.

</details>

<details><summary><b>Expected output</b></summary>

```text
result of SELECT 1
connection orders closed
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import closing

with closing(Connection("orders")) as conn:
    print(conn.query("SELECT 1"))
```

`closing` is a tiny context manager: its `__enter__` returns the object and its `__exit__` calls `.close()` on it. It adapts any object with a `close()` method to the `with` protocol, without touching the class.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Connection:
    def __init__(self, name):
        self.name = name

    def query(self, sql):
        return f"result of {sql}"

    def close(self):
        print(f"connection {self.name} closed")

    def __enter__(self):
        return self

    def __exit__(self, *exc_info):
        self.close()
        return False

with Connection("orders") as conn:
    print(conn.query("SELECT 1"))
```

When you own the class, seniors add `__enter__`/`__exit__` so callers can write `with Connection(...)` directly. `closing` is for objects you don't own, such as older library classes, generators and `urllib` responses.

</details>

In [ ]:
class Connection:
    def __init__(self, name):
        self.name = name

    def query(self, sql):
        return f"result of {sql}"

    def close(self):
        print(f"connection {self.name} closed")

# your code here


### NB24-Q13 · L3 · Patterns: temporary changes · Write

Write a `@contextmanager` called `override(settings, key, value)` that
sets `settings[key] = value` for the duration of the block and puts the
old value back afterwards, **even if the block raises**. Run the script.

```py
settings = {"debug": False, "theme": "light"}
with override(settings, "debug", True):
    print("inside:", settings)
print("after:", settings)
```

```text
inside: {'debug': True, 'theme': 'light'}
after: {'debug': False, 'theme': 'light'}
```

<details><summary><b>Hint</b></summary>

Save the old value before changing it. Restore it in a `finally` after the `yield`.

</details>

<details><summary><b>Expected output</b></summary>

```text
inside: {'debug': True, 'theme': 'light'}
after: {'debug': False, 'theme': 'light'}
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def override(settings, key, value):
    old = settings[key]
    settings[key] = value
    try:
        yield
    finally:
        settings[key] = old

settings = {"debug": False, "theme": "light"}
with override(settings, "debug", True):
    print("inside:", settings)
print("after:", settings)
```

Save, change, `yield`, restore. The `finally` is what makes this safe: if code in the block raises, the setting still goes back. Without it, one failing test could leave `debug` switched on for every test after it.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def override(settings, key, value):
    old = settings[key]
    settings[key] = value
    try:
        yield settings
    finally:
        settings[key] = old

settings = {"debug": False, "theme": "light"}
with override(settings, "debug", True):
    print("inside:", settings)
print("after:", settings)
```

Yielding `settings` lets callers write `as s` if they want. This save/change/restore shape is everywhere: `unittest.mock.patch` (NB28), `decimal.localcontext` and `redirect_stdout` all work this way. Q18 handles keys that didn't exist before.

</details>

In [ ]:
# your code here


### NB24-Q14 · L3 · Exception info · Write

Write your own version of `suppress`: a class `Ignore(*types)`. When the
block raises an exception that is an instance of one of `types`,
`__exit__` prints `ignored <ExcName>: <message>` and swallows it. Any
other exception passes through untouched. Run the script.

```py
with Ignore(ValueError, LookupError):
    int("twelve")
with Ignore(ValueError, LookupError):
    {"a": 1}["b"]
try:
    with Ignore(ValueError, LookupError):
        1 / 0
except ZeroDivisionError:
    print("ZeroDivisionError reached the caller")
```

```text
ignored ValueError: invalid literal for int() with base 10: 'twelve'
ignored KeyError: 'b'
ZeroDivisionError reached the caller
```

<details><summary><b>Hint</b></summary>

`*types` arrives as a tuple, and `isinstance(exc, types)` accepts a tuple. When there's no exception, `exc` is `None`, so check for that first. `KeyError` is a kind of `LookupError` (NB12).

</details>

<details><summary><b>Expected output</b></summary>

```text
ignored ValueError: invalid literal for int() with base 10: 'twelve'
ignored KeyError: 'b'
ZeroDivisionError reached the caller
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Ignore:
    def __init__(self, *types):
        self.types = types

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc is not None and isinstance(exc, self.types):
            print(f"ignored {exc_type.__name__}: {exc}")
            return True
        return False

with Ignore(ValueError, LookupError):
    int("twelve")
with Ignore(ValueError, LookupError):
    {"a": 1}["b"]
try:
    with Ignore(ValueError, LookupError):
        1 / 0
except ZeroDivisionError:
    print("ZeroDivisionError reached the caller")
```

`__exit__` returns `True` only for the listed types, so only those are swallowed. `isinstance` respects the exception hierarchy: `KeyError` is caught by `LookupError`. `ZeroDivisionError` isn't listed, so `__exit__` returns `False` and it reaches the caller's `except`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Ignore:
    def __init__(self, *types):
        self.types = types

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc_type is not None and issubclass(exc_type, self.types):
            print(f"ignored {exc_type.__name__}: {exc}")
            return True
        return False

with Ignore(ValueError, LookupError):
    int("twelve")
with Ignore(ValueError, LookupError):
    {"a": 1}["b"]
try:
    with Ignore(ValueError, LookupError):
        1 / 0
except ZeroDivisionError:
    print("ZeroDivisionError reached the caller")
```

`issubclass(exc_type, ...)` is how `contextlib.suppress` itself checks. In real code, use `suppress`; writing it once shows that it's only a dozen lines. Printing what was ignored is a good habit: silent suppression is how bugs hide.

</details>

In [ ]:
# your code here


### NB24-Q15 · L3 · Parenthesised multi-item with · Write

Copy `names.txt` into `upper.txt` with every line in upper case. Open
**both** files in one `with` statement, using the bracketed form that
puts each manager on its own line (Python 3.10+). Then print the new
file's contents.

```text
ANN
BOB
CY
```

Call `tmp.cleanup()` at the end to delete the temporary folder.

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
folder = Path(tmp.name)
(folder / "names.txt").write_text("ann\nbob\ncy\n", encoding="utf-8")
```

<details><summary><b>Hint</b></summary>

`with (` on the first line, then one `open(...) as name,` per line, then `):`. Loop over the source file line by line and write each line upper-cased.

</details>

<details><summary><b>Expected output</b></summary>

```text
ANN
BOB
CY
```

</details>

<details><summary><b>Solution</b></summary>

```python
with (
    open(folder / "names.txt", encoding="utf-8") as src,
    open(folder / "upper.txt", "w", encoding="utf-8") as dst,
):
    for line in src:
        dst.write(line.upper())

print((folder / "upper.txt").read_text(encoding="utf-8"), end="")
tmp.cleanup()
```

Several managers in one `with` behave like nested `with` blocks: entered left to right, exited right to left. The brackets let you put each on its own line with a trailing comma, instead of one very long line or a backslash. Both files are closed before `read_text` runs, so the written data is on disk.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
with (
    open(folder / "names.txt", encoding="utf-8") as src,
    open(folder / "upper.txt", "w", encoding="utf-8") as dst,
):
    dst.writelines(line.upper() for line in src)

print((folder / "upper.txt").read_text(encoding="utf-8"), end="")
tmp.cleanup()
```

`writelines` takes any iterable, so a generator expression streams the file without loading it all. In real code the temp folder itself would be a third manager: `tempfile.TemporaryDirectory() as tmp` in the same bracketed `with`.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
folder = Path(tmp.name)
(folder / "names.txt").write_text("ann\nbob\ncy\n", encoding="utf-8")

# your code here


### NB24-Q16 · L3 · Parenthesised multi-item with · Predict

Two `with` statements, each with two managers. In the second, `D` fails
while opening. Predict the output.

**What does this print?**

```python
class Res:
    def __init__(self, name, fail=False):
        self.name = name
        self.fail = fail

    def __enter__(self):
        if self.fail:
            raise OSError(f"{self.name} unavailable")
        print("enter", self.name)
        return self

    def __exit__(self, exc_type, exc, tb):
        print("exit", self.name)

with Res("A"), Res("B"):
    print("body")

try:
    with Res("C"), Res("D", fail=True):
        print("body never runs")
except OSError:
    print("D failed to open")
```

<details><summary><b>Hint</b></summary>

`with A, B:` is the same as `with A:` containing `with B:`. If `B` never enters, should `B` exit? Should `A`?

</details>

<details><summary><b>Expected output</b></summary>

```text
enter A
enter B
body
exit B
exit A
enter C
exit C
D failed to open
```

</details>

<details><summary><b>Solution</b></summary>

Managers enter left to right and exit in reverse: `B` before `A`, like closing nested brackets. In the second `with`, `C` enters, then `D.__enter__` raises. `D` never entered, so its `__exit__` doesn't run, but `C` did, so `exit C` runs before the error reaches the `except`.

</details>

<details><summary><b>Senior dev solution</b></summary>

This guarantee is why seniors acquire several resources in one `with` rather than by hand: whatever was successfully opened is always closed, in reverse order, even if a later one fails.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB24-Q17 · L3 · contextmanager · Refactor

Both functions repeat the same lock/unlock `try`/`finally`. Move it into a
`@contextmanager` called `locked()` and use `with locked():` in both. The
output must not change.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
log = []

def save_order(order_id):
    log.append("lock")
    try:
        print("saving", order_id)
    finally:
        log.append("unlock")

def cancel_order(order_id):
    log.append("lock")
    try:
        print("cancelling", order_id)
    finally:
        log.append("unlock")

save_order(17)
cancel_order(18)
print(log)
```

<details><summary><b>Hint</b></summary>

Everything before `try` goes before `yield`; the `finally` part goes in the context manager's own `finally`.

</details>

<details><summary><b>Expected output</b></summary>

```text
saving 17
cancelling 18
['lock', 'unlock', 'lock', 'unlock']
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import contextmanager

log = []

@contextmanager
def locked():
    log.append("lock")
    try:
        yield
    finally:
        log.append("unlock")

def save_order(order_id):
    with locked():
        print("saving", order_id)

def cancel_order(order_id):
    with locked():
        print("cancelling", order_id)

save_order(17)
cancel_order(18)
print(log)
```

A `try`/`finally` pattern that repeats is a context manager waiting to be written. Now each function holds only its own work, and the lock/unlock rule lives in one place, so a future change (say, adding a timeout) happens once.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from contextlib import contextmanager

log = []

@contextmanager
def locked():
    log.append("lock")
    try:
        yield
    finally:
        log.append("unlock")

@locked()
def save_order(order_id):
    print("saving", order_id)

@locked()
def cancel_order(order_id):
    print("cancelling", order_id)

save_order(17)
cancel_order(18)
print(log)
```

A `@contextmanager` object also works as a **decorator** (Q23): the whole function body runs inside the `with`. Seniors use that when the lock should cover the entire function, and `with` when it covers only part of it.

</details>

In [ ]:
log = []

def save_order(order_id):
    log.append("lock")
    try:
        print("saving", order_id)
    finally:
        log.append("unlock")

def cancel_order(order_id):
    log.append("lock")
    try:
        print("cancelling", order_id)
    finally:
        log.append("unlock")

save_order(17)
cancel_order(18)
print(log)


---

## L4 · Somewhat Difficult

### NB24-Q18 · L4 · Patterns: temporary changes · Write

Write `@contextmanager temporarily(settings, **changes)` that applies
**several** changes at once and undoes all of them afterwards, even if
the block raises:

- keys that existed get their old value back;
- keys that did **not** exist before are deleted again.

Run the script.

```py
settings = {"debug": False, "level": "INFO"}
with temporarily(settings, debug=True, retries=5):
    print("inside:", settings)
print("after:", settings)
try:
    with temporarily(settings, level="DEBUG"):
        print("inside:", settings)
        raise RuntimeError("test failed")
except RuntimeError:
    print("after error:", settings)
```

```text
inside: {'debug': True, 'level': 'INFO', 'retries': 5}
after: {'debug': False, 'level': 'INFO'}
inside: {'debug': False, 'level': 'DEBUG'}
after error: {'debug': False, 'level': 'INFO'}
```

<details><summary><b>Hint</b></summary>

Before changing anything, remember the old values of keys that exist and the names of keys that don't. In `finally`, put the old values back and `del` the new keys.

</details>

<details><summary><b>Expected output</b></summary>

```text
inside: {'debug': True, 'level': 'INFO', 'retries': 5}
after: {'debug': False, 'level': 'INFO'}
inside: {'debug': False, 'level': 'DEBUG'}
after error: {'debug': False, 'level': 'INFO'}
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def temporarily(settings, **changes):
    old_values = {}
    new_keys = []
    for key in changes:
        if key in settings:
            old_values[key] = settings[key]
        else:
            new_keys.append(key)
    settings.update(changes)
    try:
        yield settings
    finally:
        settings.update(old_values)
        for key in new_keys:
            del settings[key]

settings = {"debug": False, "level": "INFO"}
with temporarily(settings, debug=True, retries=5):
    print("inside:", settings)
print("after:", settings)
try:
    with temporarily(settings, level="DEBUG"):
        print("inside:", settings)
        raise RuntimeError("test failed")
except RuntimeError:
    print("after error:", settings)
```

Restoring needs two different actions. An old key gets its old value back, but a new key has no "old value": setting it to `None` would leave `retries: None` behind. So the two kinds of key are tracked separately, and `finally` undoes both whether the block succeeded or raised. Keys updated in place keep their original position in the dict.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from contextlib import contextmanager

_MISSING = object()

@contextmanager
def temporarily(settings, **changes):
    saved = {key: settings.get(key, _MISSING) for key in changes}
    settings.update(changes)
    try:
        yield settings
    finally:
        for key, old in saved.items():
            if old is _MISSING:
                del settings[key]
            else:
                settings[key] = old

settings = {"debug": False, "level": "INFO"}
with temporarily(settings, debug=True, retries=5):
    print("inside:", settings)
print("after:", settings)
try:
    with temporarily(settings, level="DEBUG"):
        print("inside:", settings)
        raise RuntimeError("test failed")
except RuntimeError:
    print("after error:", settings)
```

A private *sentinel* object (`_MISSING = object()`) marks "there was no value". Unlike `None`, it can never be a real setting, so a key whose old value really was `None` is restored correctly. `unittest.mock.patch.dict` (NB28) does this job in tests.

</details>

In [ ]:
# your code here


### NB24-Q19 · L4 · Patterns: transactions · Write

Write `@contextmanager atomic(items)` for a list. It yields a **copy**
(the draft) to work on. If the block finishes, the original list's
contents are replaced with the draft's. If the block raises, the draft is
thrown away and the exception carries on. Run the script.

```py
cart = ["apple"]
with atomic(cart) as draft:
    draft.append("bread")
    draft.append("milk")
print(cart)
try:
    with atomic(cart) as draft:
        draft.append("caviar")
        raise ValueError("card declined")
except ValueError as e:
    print("failed:", e)
print(cart)
```

```text
['apple', 'bread', 'milk']
failed: card declined
['apple', 'bread', 'milk']
```

<details><summary><b>Hint</b></summary>

Make the copy, `yield` it, and only **after** the `yield` (which is skipped when the block raises) copy it back with `items[:] = draft`. No `try` needed: you *want* the failure path to skip the commit.

</details>

<details><summary><b>Expected output</b></summary>

```text
['apple', 'bread', 'milk']
failed: card declined
['apple', 'bread', 'milk']
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def atomic(items):
    draft = items.copy()
    yield draft
    items[:] = draft

cart = ["apple"]
with atomic(cart) as draft:
    draft.append("bread")
    draft.append("milk")
print(cart)
try:
    with atomic(cart) as draft:
        draft.append("caviar")
        raise ValueError("card declined")
except ValueError as e:
    print("failed:", e)
print(cart)
```

This is the one case where code after `yield` should **not** be in `finally`: the commit must run only on success. When the block raises, `@contextmanager` re-raises at the `yield`, the commit line is skipped, and the exception continues to the caller. `items[:] = draft` replaces the contents in place, so every variable that refers to `cart` sees the change.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import copy
from contextlib import contextmanager

@contextmanager
def atomic(items):
    draft = copy.deepcopy(items)
    yield draft
    items[:] = draft  # commit only on success; an exception skips this line

cart = ["apple"]
with atomic(cart) as draft:
    draft.extend(["bread", "milk"])
print(cart)
try:
    with atomic(cart) as draft:
        draft.append("caviar")
        raise ValueError("card declined")
except ValueError as e:
    print(f"failed: {e}")
print(cart)
```

`deepcopy` (NB05) protects nested data too: with a shallow copy, editing a dict *inside* the draft would change the original immediately. The comment matters, because a reviewer trained on Q11 might "fix" this into a `finally` and commit failed work.

</details>

In [ ]:
# your code here


### NB24-Q20 · L4 · ExitStack · Write

You need to hold open a **variable** number of resources, so you can't
write them all in one `with` line. Use `contextlib.ExitStack` to enter
a `Res` for every name in `names`, print `using db, cache, queue`, and
let the stack close them all.

```text
enter db
enter cache
enter queue
using db, cache, queue
exit queue
exit cache
exit db
```

**Starting code (already in the cell below):**

```python
class Res:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print("enter", self.name)
        return self

    def __exit__(self, exc_type, exc, tb):
        print("exit", self.name)

names = ["db", "cache", "queue"]
```

<details><summary><b>Hint</b></summary>

`with ExitStack() as stack:` then, in a loop, `stack.enter_context(Res(name))`. It enters the manager and returns what `__enter__` returned.

</details>

<details><summary><b>Expected output</b></summary>

```text
enter db
enter cache
enter queue
using db, cache, queue
exit queue
exit cache
exit db
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import ExitStack

with ExitStack() as stack:
    resources = []
    for name in names:
        resources.append(stack.enter_context(Res(name)))
    print("using", ", ".join(r.name for r in resources))
```

`ExitStack` is a context manager that holds other context managers. `enter_context` enters each one and pushes its exit onto a stack. When the `with ExitStack()` block ends, the stack unwinds last-in, first-out, exactly as if you'd written three nested `with` blocks, but for any number of resources.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from contextlib import ExitStack

with ExitStack() as stack:
    resources = [stack.enter_context(Res(name)) for name in names]
    print(f"using {', '.join(r.name for r in resources)}")
```

A list comprehension is safe here: if the third `Res` fails to enter, the two already entered are on the stack and still get closed. Seniors reach for `ExitStack` whenever the number of resources is only known at run time, such as files from a folder listing.

</details>

In [ ]:
class Res:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print("enter", self.name)
        return self

    def __exit__(self, exc_type, exc, tb):
        print("exit", self.name)

names = ["db", "cache", "queue"]

# your code here


### NB24-Q21 · L4 · contextmanager · Error

This generator yields twice. What is printed, and which error is raised?

**Which error does this raise?**

```python
from contextlib import contextmanager

@contextmanager
def session():
    print("start")
    yield "s1"
    print("between")
    yield "s2"

with session() as s:
    print("using", s)
```

<details><summary><b>Hint</b></summary>

On exit, `@contextmanager` resumes the generator once and expects it to **finish**. What happens instead?

</details>

<details><summary><b>Expected output</b></summary>

```text
start
using s1
between
RuntimeError
```

</details>

<details><summary><b>Solution</b></summary>

`RuntimeError`: on exit, the manager resumes the generator expecting it to end. It prints `between` and then reaches a second `yield`, so the generator didn't stop. `@contextmanager` raises `RuntimeError` ("generator didn't stop"). A `@contextmanager` generator must yield **exactly once**.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def session():
    print("start")
    try:
        yield "s1"
    finally:
        print("between")

with session() as s:
    print("using", s)
```

If you need two phases, use two `with` blocks or a class with explicit methods. Putting a `yield` inside a loop in a `@contextmanager` is the usual way to hit this by accident.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB24-Q22 · L4 · Exception suppression · Fix

`report_errors` should print a friendly message and then **let the error
continue**, so the caller knows the save failed. Instead, `save_all`
carries on and wrongly prints `all saved`. Fix it so the output is:

```text
save failed: disk full
caller knows it failed
```

**Buggy code (copied into the cell below):**

```python
from contextlib import contextmanager

@contextmanager
def report_errors(task):
    try:
        yield
    except OSError as e:
        print(f"{task} failed: {e}")

def save_all():
    with report_errors("save"):
        raise OSError("disk full")
    print("all saved")

try:
    save_all()
except OSError:
    print("caller knows it failed")
```

<details><summary><b>Hint</b></summary>

Inside a `@contextmanager`, catching the exception at the `yield` and not raising it again means the same as `__exit__` returning `True`.

</details>

<details><summary><b>Expected output</b></summary>

```text
save failed: disk full
caller knows it failed
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def report_errors(task):
    try:
        yield
    except OSError as e:
        print(f"{task} failed: {e}")
        raise

def save_all():
    with report_errors("save"):
        raise OSError("disk full")
    print("all saved")

try:
    save_all()
except OSError:
    print("caller knows it failed")
```

The block's exception is re-raised inside the generator at `yield`. Catching it there and finishing normally tells `@contextmanager` it was handled, so it's **suppressed**: `save_all` continued and claimed success. A bare `raise` after printing sends the same exception on to the caller.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from contextlib import contextmanager

@contextmanager
def report_errors(task):
    try:
        yield
    except OSError as e:
        e.add_note(f"while running task {task!r}")
        print(f"{task} failed: {e}")
        raise

def save_all():
    with report_errors("save"):
        raise OSError("disk full")
    print("all saved")

try:
    save_all()
except OSError:
    print("caller knows it failed")
```

Suppression by accident is the most dangerous context-manager bug: no crash, wrong result. Seniors also attach context to the error with `add_note` (NB12), which shows up in the traceback without changing the exception type the caller catches.

</details>

In [ ]:
from contextlib import contextmanager

@contextmanager
def report_errors(task):
    try:
        yield
    except OSError as e:
        print(f"{task} failed: {e}")

def save_all():
    with report_errors("save"):
        raise OSError("disk full")
    print("all saved")

try:
    save_all()
except OSError:
    print("caller knows it failed")


### NB24-Q23 · L4 · contextmanager · Predict

A `@contextmanager` object used as a **decorator**. Predict the output.

**What does this print?**

```python
from contextlib import contextmanager

@contextmanager
def section(name):
    print(f"[{name}]")
    try:
        yield
    finally:
        print(f"[/{name}]")

@section("job")
def run(n):
    print("run", n)

run(1)
run(2)
```

<details><summary><b>Hint</b></summary>

Context managers made by `@contextmanager` can also decorate a function. Does the section open once, when the decorator is applied, or on every call?

</details>

<details><summary><b>Expected output</b></summary>

```text
[job]
run 1
[/job]
[job]
run 2
[/job]
```

</details>

<details><summary><b>Solution</b></summary>

Managers made with `@contextmanager` are also *context decorators*: decorating `run` wraps every **call** in `with section("job"):`. Each call gets a fresh generator, so `[job]` and `[/job]` appear around both runs, and nothing prints at decoration time.

</details>

<details><summary><b>Senior dev solution</b></summary>

This is the bridge between NB23 and this notebook: one object gives you both `with section("x"):` for part of a function and `@section("x")` for all of it. Class-based managers get the same power by inheriting from `contextlib.ContextDecorator`.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB24-Q24 · L4 · The with protocol · Write

Write a **reusable, nestable** class `Indenter`. Each `with ind:` goes
one level deeper, and leaving the block goes back up. Its method
`say(text)` prints the text indented by two spaces per level, where the
outermost `with` is level 0. Run the script.

```py
ind = Indenter()
with ind:
    ind.say("Shop")
    with ind:
        ind.say("Fruit")
        with ind:
            ind.say("apple")
        ind.say("Veg")
    ind.say("Total: 3")
```

```text
Shop
  Fruit
    apple
  Veg
Total: 3
```

<details><summary><b>Hint</b></summary>

Keep a `level` counter on the object, starting at `-1`. `__enter__` adds 1 and returns `self`; `__exit__` subtracts 1. The same object is entered several times.

</details>

<details><summary><b>Expected output</b></summary>

```text
Shop
  Fruit
    apple
  Veg
Total: 3
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Indenter:
    def __init__(self):
        self.level = -1

    def __enter__(self):
        self.level += 1
        return self

    def __exit__(self, exc_type, exc, tb):
        self.level -= 1

    def say(self, text):
        print("  " * self.level + text)

ind = Indenter()
with ind:
    ind.say("Shop")
    with ind:
        ind.say("Fruit")
        with ind:
            ind.say("apple")
        ind.say("Veg")
    ind.say("Total: 3")
```

One object, entered three times. The state lives on the object, so each `__enter__` deepens the indent and each `__exit__` undoes exactly its own step. A manager that can be entered again while it's already active is called *reentrant*. Many (like an open file) are single-use instead.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Indenter:
    def __init__(self, width=2):
        self.width = width
        self.level = -1

    def __enter__(self):
        self.level += 1
        return self

    def __exit__(self, *exc_info):
        self.level -= 1
        return False

    def say(self, text):
        print(f"{' ' * (self.width * self.level)}{text}")

ind = Indenter()
with ind:
    ind.say("Shop")
    with ind:
        ind.say("Fruit")
        with ind:
            ind.say("apple")
        ind.say("Veg")
    ind.say("Total: 3")
```

The indent width becomes a setting instead of a magic number. Because `__exit__` always runs, an exception deep inside still leaves the level correct for the next output: state that is undone in `__exit__` can never drift.

</details>

In [ ]:
# your code here


---

## L5 · Hard

### NB24-Q25 · L5 · ExitStack · Write

Write `open_all(folder, names)` that opens every file for writing and
returns `(files, stack)`. Use an `ExitStack`:

- after opening each file, print `opened <name>` and register a callback
  that prints `closed <name>`;
- if **any** open fails, every file opened so far must be closed (and the
  error passed on);
- on success, hand the open files to the caller with `stack.pop_all()`,
  so they stay open after `open_all` returns.

Then run the script (both lists use the given `folder`; `missing/` does
not exist).

```py
files, stack = open_all(folder, ["a.txt", "b.txt", "c.txt"])
print("got", len(files), "open files")
with stack:
    for f in files:
        f.write("ok\n")
try:
    open_all(folder, ["x.txt", "missing/y.txt", "z.txt"])
except FileNotFoundError:
    print("FileNotFoundError: nothing left open")
tmp.cleanup()
```

```text
opened a.txt
opened b.txt
opened c.txt
got 3 open files
closed c.txt
closed b.txt
closed a.txt
opened x.txt
closed x.txt
FileNotFoundError: nothing left open
```

**Starting code (already in the cell below):**

```python
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
folder = Path(tmp.name)
```

<details><summary><b>Hint</b></summary>

Do everything inside `with ExitStack() as stack:`. If the loop raises, the `with` unwinds the stack for you. If it finishes, `stack.pop_all()` returns a *new* stack holding all the registered exits and leaves the old one empty, so leaving the `with` closes nothing.

</details>

<details><summary><b>Expected output</b></summary>

```text
opened a.txt
opened b.txt
opened c.txt
got 3 open files
closed c.txt
closed b.txt
closed a.txt
opened x.txt
closed x.txt
FileNotFoundError: nothing left open
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import ExitStack

def open_all(folder, names):
    with ExitStack() as stack:
        files = []
        for name in names:
            f = stack.enter_context(open(folder / name, "w", encoding="utf-8"))
            print("opened", name)
            stack.callback(print, "closed", name)
            files.append(f)
        return files, stack.pop_all()

files, stack = open_all(folder, ["a.txt", "b.txt", "c.txt"])
print("got", len(files), "open files")
with stack:
    for f in files:
        f.write("ok\n")
try:
    open_all(folder, ["x.txt", "missing/y.txt", "z.txt"])
except FileNotFoundError:
    print("FileNotFoundError: nothing left open")
tmp.cleanup()
```

`stack.callback(func, *args)` registers any function to run on exit, not only context managers. When `missing/y.txt` fails, the exception leaves the `with ExitStack()` block, which closes `x.txt` and passes the error on. On success, `pop_all()` moves every pending exit to a new stack, so the original closes nothing and the caller now owns the cleanup. The caller's `with stack:` closes them later, in reverse order.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from contextlib import ExitStack

def open_all(folder, names):
    with ExitStack() as stack:
        files = []
        for name in names:
            files.append(stack.enter_context(open(folder / name, "w", encoding="utf-8")))
            print(f"opened {name}")
            stack.callback(print, f"closed {name}")
        return files, stack.pop_all()

files, stack = open_all(folder, ["a.txt", "b.txt", "c.txt"])
print(f"got {len(files)} open files")
with stack:
    for f in files:
        f.write("ok\n")
try:
    open_all(folder, ["x.txt", "missing/y.txt", "z.txt"])
except FileNotFoundError:
    print("FileNotFoundError: nothing left open")
tmp.cleanup()
```

"Acquire all or nothing, then transfer ownership with `pop_all()`" is the pattern from the `contextlib` docs, and seniors use it for opening a set of files, connections or locks together. Returning the stack makes ownership explicit: whoever holds it must close it.

</details>

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
folder = Path(tmp.name)

# your code here


### NB24-Q26 · L5 · Patterns: timers · Write

Write a class `Profiler(clock)` with a `@contextmanager` **method**
`section(name)`. It times each section with `clock` and stores the
seconds in the dict `self.times`. Sections can nest, and a nested
section's key is its full path joined with `/` (like `load/parse`). A
section that raises must **still** be recorded, and the error must
still reach the caller. Run the script, then print every entry with 2
decimal places.

```py
prof = Profiler(fake_clock)
with prof.section("load"):
    with prof.section("parse"):
        pass
    with prof.section("validate"):
        pass
try:
    with prof.section("save"):
        raise OSError("disk full")
except OSError:
    print("save failed")
```

```text
save failed
load/parse: 0.50s
load/validate: 1.50s
load: 3.00s
save: 0.25s
```

**Starting code (already in the cell below):**

```python
readings = iter([0.0, 0.5, 1.0, 1.0, 2.5, 3.0, 10.0, 10.25])

def fake_clock():
    return next(readings)
```

<details><summary><b>Hint</b></summary>

Keep a list `self.path` of the open section names. On entry, append the name and read the clock. In `finally`, read the clock again, store the time under `"/".join(self.path)`, then pop the name.

</details>

<details><summary><b>Expected output</b></summary>

```text
save failed
load/parse: 0.50s
load/validate: 1.50s
load: 3.00s
save: 0.25s
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import contextmanager

class Profiler:
    def __init__(self, clock):
        self.clock = clock
        self.times = {}
        self.path = []

    @contextmanager
    def section(self, name):
        self.path.append(name)
        start = self.clock()
        try:
            yield
        finally:
            self.times["/".join(self.path)] = self.clock() - start
            self.path.pop()

prof = Profiler(fake_clock)
with prof.section("load"):
    with prof.section("parse"):
        pass
    with prof.section("validate"):
        pass
try:
    with prof.section("save"):
        raise OSError("disk full")
except OSError:
    print("save failed")

for name, seconds in prof.times.items():
    print(f"{name}: {seconds:.2f}s")
```

`self.path` acts as a stack of open sections, so the key for `parse` is built while `load` is still open. Inner sections finish first, so they are added to `times` before `load`. The `finally` records `save` even though it raised, and since nothing catches the error inside the generator, it still reaches the caller. `@contextmanager` works on methods like any decorator.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from contextlib import contextmanager

class Profiler:
    def __init__(self, clock):
        self.clock = clock
        self.times = {}
        self._path = []

    @contextmanager
    def section(self, name):
        self._path.append(name)
        key = "/".join(self._path)
        start = self.clock()
        try:
            yield self
        finally:
            self.times[key] = self.times.get(key, 0) + (self.clock() - start)
            self._path.pop()

prof = Profiler(fake_clock)
with prof.section("load"):
    with prof.section("parse"):
        pass
    with prof.section("validate"):
        pass
try:
    with prof.section("save"):
        raise OSError("disk full")
except OSError:
    print("save failed")

for name, seconds in prof.times.items():
    print(f"{name}: {seconds:.2f}s")
```

The senior adds times up, so a section run in a loop reports its total rather than only its last run, and computes the key on entry. `_path` signals "internal". Real profilers (`cProfile`, NB29) do the same bookkeeping for every function automatically.

</details>

In [ ]:
readings = iter([0.0, 0.5, 1.0, 1.0, 2.5, 3.0, 10.0, 10.25])

def fake_clock():
    return next(readings)

# your code here


### NB24-Q27 · L5 · Exception suppression · Predict

Nested managers, one of them `suppress`. Predict the output, line by
line.

**What does this print?**

```python
from contextlib import contextmanager, suppress

@contextmanager
def step(name):
    print("enter", name)
    try:
        yield
    finally:
        print("exit", name)

with step("outer"):
    with suppress(KeyError), step("inner"):
        print("lookup")
        {}["sku-42"]
        print("not reached")
    print("outer continues")
print("done")
```

<details><summary><b>Hint</b></summary>

`with suppress(KeyError), step("inner"):` enters `suppress` first, so it exits **last**. Which manager does the `KeyError` pass through first, and which one stops it?

</details>

<details><summary><b>Expected output</b></summary>

```text
enter outer
enter inner
lookup
exit inner
outer continues
exit outer
done
```

</details>

<details><summary><b>Solution</b></summary>

The `KeyError` skips `not reached` and passes through `step("inner")` first: its `finally` prints `exit inner` but doesn't stop the error. Next it reaches `suppress`, which swallows it. So the `outer` block carries on normally, prints `outer continues`, exits, and the program prints `done`.

</details>

<details><summary><b>Senior dev solution</b></summary>

Managers to the **left** in one `with` are outside the ones to the right. Putting `suppress` on the left means everything to its right still cleans up before the error is swallowed. Swap them and `step("inner")` would see no exception at all.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB24-Q28 · L5 · ExitStack · Refactor

`run` opens three resources with a pyramid of nested `try`/`finally`
blocks. Refactor it with `ExitStack` so it works for **any** list of
names. Keep `Res` as it is and keep the output identical.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
class Res:
    def __init__(self, name):
        self.name = name

    def open(self):
        print("open", self.name)

    def close(self):
        print("close", self.name)

def run():
    db = Res("db")
    db.open()
    try:
        cache = Res("cache")
        cache.open()
        try:
            queue = Res("queue")
            queue.open()
            try:
                print("working")
            finally:
                queue.close()
        finally:
            cache.close()
    finally:
        db.close()

run()
```

<details><summary><b>Hint</b></summary>

Give `run` a `names` parameter. In a loop: create the resource, open it, then register its `close` with `stack.callback(...)`.

</details>

<details><summary><b>Expected output</b></summary>

```text
open db
open cache
open queue
working
close queue
close cache
close db
```

</details>

<details><summary><b>Solution</b></summary>

```python
from contextlib import ExitStack

class Res:
    def __init__(self, name):
        self.name = name

    def open(self):
        print("open", self.name)

    def close(self):
        print("close", self.name)

def run(names):
    with ExitStack() as stack:
        for name in names:
            res = Res(name)
            res.open()
            stack.callback(res.close)
        print("working")

run(["db", "cache", "queue"])
```

Each `try`/`finally` level in the pyramid becomes one `callback`, registered right after a successful `open()`. The stack runs them in reverse order, which is exactly what the nesting did, and only for resources that actually opened. The code no longer grows sideways with each new resource.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from contextlib import ExitStack

class Res:
    def __init__(self, name):
        self.name = name

    def open(self):
        print("open", self.name)

    def close(self):
        print("close", self.name)

def run(names):
    with ExitStack() as stack:
        for name in names:
            res = Res(name)
            res.open()
            stack.callback(res.close)
        print("working")

run(("db", "cache", "queue"))
```

Almost the same code, which is the point: there's very little left to improve. The one rule seniors watch is order: register the cleanup **after** `open()` succeeds, never before, or a failed open would try to close something that never opened. A tuple signals the list of names is fixed.

</details>

In [ ]:
class Res:
    def __init__(self, name):
        self.name = name

    def open(self):
        print("open", self.name)

    def close(self):
        print("close", self.name)

def run():
    db = Res("db")
    db.open()
    try:
        cache = Res("cache")
        cache.open()
        try:
            queue = Res("queue")
            queue.open()
            try:
                print("working")
            finally:
                queue.close()
        finally:
            cache.close()
    finally:
        db.close()

run()


---

## Mini-project

Pull together everything from this notebook.

## NB24-P · L5 · Mini-project: Transaction Manager · Write

Build a tiny database with transactions that roll back on error.

1. `Database(data)` stores a dict in `self.data`.
2. Its method `transaction(name)` is a `@contextmanager`. On entry it
   takes a deep copy of `self.data` (a snapshot). If the block finishes,
   it prints `<name>: committed`. If the block raises, it restores the
   snapshot **into the same dict** (so anyone holding `db.data` sees it),
   prints `<name>: rolled back (<ExcName>: <message>)`, and re-raises.
3. Transactions can nest: an inner one that fails rolls back only its own
   changes, and the outer one can catch the error and still commit.
4. Write `transfer(db, src, dst, amount)` that runs inside
   `db.transaction(f"transfer {src}->{dst}")`: subtract from `src`, raise
   `ValueError(f"{src} would be overdrawn")` if `src` goes below 0, then add
   to `dst` (an unknown `dst` raises `KeyError` naturally).

Run the script below the expected output.

```text
transfer ann->bob: committed
{'ann': 70, 'bob': 80}
transfer bob->ann: rolled back (ValueError: bob would be overdrawn)
{'ann': 70, 'bob': 80}
transfer ann->cy: rolled back (KeyError: 'cy')
{'ann': 70, 'bob': 80}
bonus: rolled back (ValueError: bonus not approved)
batch: committed
{'ann': 65, 'bob': 80}
```

Script:

```py
db = Database({"ann": 100, "bob": 50})
for src, dst, amount in [("ann", "bob", 30), ("bob", "ann", 500), ("ann", "cy", 10)]:
    try:
        transfer(db, src, dst, amount)
    except (ValueError, KeyError):
        pass
    print(db.data)

with db.transaction("batch"):
    db.data["ann"] -= 5
    try:
        with db.transaction("bonus"):
            db.data["bob"] += 1000
            raise ValueError("bonus not approved")
    except ValueError:
        pass
print(db.data)
```

<details><summary><b>Hint</b></summary>

`try: yield` / `except Exception as e:` restore, print, `raise` / `else:` print committed. Restore with `self.data.clear()` then `self.data.update(snapshot)`. Each `transaction` call takes its own snapshot, which is all nesting needs.

</details>

<details><summary><b>Expected output</b></summary>

```text
transfer ann->bob: committed
{'ann': 70, 'bob': 80}
transfer bob->ann: rolled back (ValueError: bob would be overdrawn)
{'ann': 70, 'bob': 80}
transfer ann->cy: rolled back (KeyError: 'cy')
{'ann': 70, 'bob': 80}
bonus: rolled back (ValueError: bonus not approved)
batch: committed
{'ann': 65, 'bob': 80}
```

</details>

<details><summary><b>Solution</b></summary>

```python
import copy
from contextlib import contextmanager

class Database:
    def __init__(self, data):
        self.data = data

    @contextmanager
    def transaction(self, name):
        snapshot = copy.deepcopy(self.data)
        try:
            yield self
        except Exception as e:
            self.data.clear()
            self.data.update(snapshot)
            print(f"{name}: rolled back ({type(e).__name__}: {e})")
            raise
        else:
            print(f"{name}: committed")

def transfer(db, src, dst, amount):
    with db.transaction(f"transfer {src}->{dst}"):
        db.data[src] -= amount
        if db.data[src] < 0:
            raise ValueError(f"{src} would be overdrawn")
        db.data[dst] += amount

db = Database({"ann": 100, "bob": 50})
for src, dst, amount in [("ann", "bob", 30), ("bob", "ann", 500), ("ann", "cy", 10)]:
    try:
        transfer(db, src, dst, amount)
    except (ValueError, KeyError):
        pass
    print(db.data)

with db.transaction("batch"):
    db.data["ann"] -= 5
    try:
        with db.transaction("bonus"):
            db.data["bob"] += 1000
            raise ValueError("bonus not approved")
    except ValueError:
        pass
print(db.data)
```

The snapshot is taken on entry, before any change. On failure, `clear()` + `update()` restores it **in place**: rebinding `self.data = snapshot` would leave anyone holding the old dict looking at the half-finished changes. The `ann->cy` transfer shows why rollback matters: `ann` had already lost 10 before `cy` turned out not to exist. The bare `raise` keeps errors visible (no accidental suppression). For nesting, `bonus` snapshots *after* the fee, so its rollback keeps the fee, and `batch` commits because the error was caught inside it.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import copy
from contextlib import contextmanager

class Database:
    def __init__(self, data):
        self.data = data

    @contextmanager
    def transaction(self, name):
        snapshot = copy.deepcopy(self.data)
        try:
            yield self
        except BaseException as e:
            self.data.clear()
            self.data.update(snapshot)
            print(f"{name}: rolled back ({type(e).__name__}: {e})")
            raise
        print(f"{name}: committed")

def transfer(db, src, dst, amount):
    with db.transaction(f"transfer {src}->{dst}"):
        db.data[src] -= amount
        if db.data[src] < 0:
            raise ValueError(f"{src} would be overdrawn")
        db.data[dst] += amount

db = Database({"ann": 100, "bob": 50})
for src, dst, amount in [("ann", "bob", 30), ("bob", "ann", 500), ("ann", "cy", 10)]:
    try:
        transfer(db, src, dst, amount)
    except (ValueError, KeyError):
        pass
    print(db.data)

with db.transaction("batch"):
    db.data["ann"] -= 5
    try:
        with db.transaction("bonus"):
            db.data["bob"] += 1000
            raise ValueError("bonus not approved")
    except ValueError:
        pass
print(db.data)
```

Catching `BaseException` (and always re-raising) means even Ctrl+C (`KeyboardInterrupt`) mid-transaction rolls back; that is one of the rare places a broad catch is right, because nothing is swallowed. Real databases do this with `BEGIN`/`COMMIT`/`ROLLBACK` and savepoints for nesting, and `sqlite3` connections (NB31) are context managers that commit or roll back exactly like this.

</details>

In [ ]:
# your code here


---

## Done

Next up: **NB25 · Descriptors Metaprogramming** in `advanced/`.